# CS672 – Project 2: Deep Learning Models for NYC Taxi Trip Duration Prediction with Enriched Meteostat Weather

## 1. Imports, runtime profile, and setup

### 1.1 Core Python and data libraries
Load base utilities used throughout preprocessing, plotting, and file handling.

In [1]:
import os
import time
import warnings
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1.2 Deep learning frameworks
Import TensorFlow/Keras for required models and PyTorch for extra credit comparison.

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, Sequential
from keras.optimizers import SGD, Adam, RMSprop
from keras.callbacks import TensorBoard

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

2026-04-15 19:32:41.660266: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 19:32:42.491039: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 19:32:42.491530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 19:32:42.628511: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 19:32:43.070892: I tensorflow/core/platform/cpu_feature_guar

### 1.3 Modeling helpers, runtime profile, and reproducibility

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from meteostat import Point, Daily

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Runtime controls: keep default on the safer local-laptop profile.
RUN_MODE = "stable"   # choose from: "fast", "stable", "full"
ENABLE_PYTORCH_EXTRA_CREDIT = False
SAVE_MODELING_ARTIFACTS = True
LOAD_READY_DATASET_IF_AVAILABLE = True
ENABLE_TENSORBOARD = False

# Assignment assumes batch_size uses default 32.
BATCH_SIZE = 32

RUN_CONFIG = {
    "fast":   {"max_rows": 20_000,  "viz_rows": 10_000,  "torch_rows": 10_000,  "epochs": 25,  "verbose": 1},
    "stable": {"max_rows": 60_000,  "viz_rows": 20_000,  "torch_rows": 20_000,  "epochs": 100, "verbose": 1},
    "full":   {"max_rows": 200_000, "viz_rows": 50_000,  "torch_rows": 50_000,  "epochs": 100, "verbose": 1},
}

if RUN_MODE not in RUN_CONFIG:
    raise ValueError(f"RUN_MODE must be one of {list(RUN_CONFIG)}, got {RUN_MODE!r}")

MAX_ROWS = RUN_CONFIG[RUN_MODE]["max_rows"]
VIZ_MAX_ROWS = RUN_CONFIG[RUN_MODE]["viz_rows"]
TORCH_MAX_ROWS = RUN_CONFIG[RUN_MODE]["torch_rows"]
DEFAULT_EPOCHS = RUN_CONFIG[RUN_MODE]["epochs"]
FIT_VERBOSE = RUN_CONFIG[RUN_MODE]["verbose"]

CACHE_DIR = Path("project2_cache")
CACHE_DIR.mkdir(exist_ok=True)

print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("RUN_MODE:", RUN_MODE)
print("MAX_ROWS:", MAX_ROWS)
print("VIZ_MAX_ROWS:", VIZ_MAX_ROWS)
print("DEFAULT_EPOCHS:", DEFAULT_EPOCHS)
print("BATCH_SIZE:", BATCH_SIZE)

TensorFlow: 2.15.0
PyTorch: 2.1.2+cu121
Pandas: 2.2.0
NumPy: 1.26.4


## 2. Load cleaned taxi data from Project 1

We reuse the **Beam-cleaned parquet shard outputs** produced in Project 1:

- `clean_jan-00000-of-00001`
- `clean_mar-00000-of-00001`
- `clean_may-00000-of-00001`

The helper below searches common locations so the notebook is easy to run inside the same container.


### 2.1 Locate cleaned parquet shards
Search common directories, prioritizing Project 1 `beam_outputs`.

In [4]:
def find_file(filename: str) -> Path:
    here = Path(".").resolve()
    search_dirs = [
        here,
        here / "beam_outputs",
        here / "Project 2",
        here.parent / "Project 1" / "beam_outputs",
        Path("./Project 1/beam_outputs"),
        Path("../Project 1/beam_outputs"),
        Path("/app/notebooks"),
        Path("/app/notebooks/../Project 1/beam_outputs"),
        Path("/mnt/data"),
    ]

    for directory in search_dirs:
        candidate = (directory / filename).resolve()
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find {filename} in: {search_dirs}")

### 2.2 Load month-level taxi datasets
Read Jan, Mar, and May cleaned outputs from Project 1.

In [5]:
jan_path = find_file("clean_jan-00000-of-00001")
mar_path = find_file("clean_mar-00000-of-00001")
may_path = find_file("clean_may-00000-of-00001")

jan = pd.read_parquet(jan_path)
mar = pd.read_parquet(mar_path)
may = pd.read_parquet(may_path)

### 2.3 Sanity check taxi inputs
Confirm paths and baseline schema before enrichment.

In [6]:
print("January path:", jan_path)
print("March path:  ", mar_path)
print("May path:    ", may_path)
print("\nShapes")
print("January:", jan.shape)
print("March:  ", mar.shape)
print("May:    ", may.shape)

display(jan.head())

January path: /app/notebooks/clean_jan-00000-of-00001
March path:   /app/notebooks/clean_mar-00000-of-00001
May path:     /app/notebooks/clean_may-00000-of-00001

Shapes
January: (6303479, 15)
March:   (2960074, 15)
May:     (336318, 15)


,VendorID,RatecodeID,PULocationID,DOLocationID,payment_type,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,pickup_hour,pickup_weekday,pickup_day,pickup_month,trip_duration_min
0,1,1.0,238,239,1,1.0,1.20,6.0,1.47,11.27,0,2,1,1,4.800000
1,1,1.0,239,238,1,1.0,1.20,7.0,1.50,12.30,0,2,1,1,7.416667
2,1,1.0,238,238,1,1.0,0.60,6.0,1.00,10.80,0,2,1,1,6.183333
3,1,1.0,238,151,1,1.0,0.80,5.5,1.36,8.16,0,2,1,1,4.850000
4,2,1.0,7,193,2,1.0,0.03,2.5,0.00,3.80,0,2,1,1,0.883333


## 3. Reconstruct dates and load weather data

The cleaned Beam files contain engineered calendar fields such as `pickup_day`, `pickup_month`, and `pickup_hour`.  
To merge with weather, we reconstruct a full `pickup_date`.

Then we download daily weather for NYC using **Meteostat**.


### 3.1 Reconstruct pickup dates
Create a stable calendar date from `pickup_day` + known month.

In [7]:
from datetime import datetime

def ensure_pickup_date(df: pd.DataFrame, month_value: int) -> pd.DataFrame:
    if "pickup_date" not in df.columns:
        day = pd.to_numeric(df["pickup_day"], errors="coerce").astype("Int64")
        ymd = (2020 * 10000 + month_value * 100) + day
        df = df.copy()
        df["pickup_date"] = pd.to_datetime(ymd.astype("string"), format="%Y%m%d", errors="coerce")
    return df

jan = ensure_pickup_date(jan, 1)
mar = ensure_pickup_date(mar, 3)
may = ensure_pickup_date(may, 5)

print("pickup_date columns ready:",
      "jan" if "pickup_date" in jan.columns else "jan-MISSING",
      "mar" if "pickup_date" in mar.columns else "mar-MISSING",
      "may" if "pickup_date" in may.columns else "may-MISSING")

pickup_date columns ready: jan mar may


### 3.2 Load daily and hourly Meteostat weather files
Use the provided NYC daily weather file for the required merge and the hourly file to engineer additional daily climate signals such as humidity, dew point, precipitation-hour frequency, and within-day temperature variability.

In [ ]:
daily_weather_candidates = [
    Path("NYC weather - Jan-May 2020 - daily.csv"),
    Path("./Project 2/NYC weather - Jan-May 2020 - daily.csv"),
    Path("../Project 2/NYC weather - Jan-May 2020 - daily.csv"),
    Path("/mnt/data/NYC weather - Jan-May 2020 - daily.csv"),
    Path("/app/notebooks/NYC weather - Jan-May 2020 - daily.csv"),
]

hourly_weather_candidates = [
    Path("NYC weather - Jan-May 2020 - hourly.csv"),
    Path("./Project 2/NYC weather - Jan-May 2020 - hourly.csv"),
    Path("../Project 2/NYC weather - Jan-May 2020 - hourly.csv"),
    Path("/mnt/data/NYC weather - Jan-May 2020 - hourly.csv"),
    Path("/app/notebooks/NYC weather - Jan-May 2020 - hourly.csv"),
]

daily_weather_path = next((p for p in daily_weather_candidates if p.exists()), None)
hourly_weather_path = next((p for p in hourly_weather_candidates if p.exists()), None)

if daily_weather_path is not None:
    daily_weather = pd.read_csv(daily_weather_path)
    print(f"Loaded daily weather from local file: {daily_weather_path}")
else:
    print("Local daily weather file not found. Fetching daily weather from Meteostat API...")
    nyc = Point(40.706, -74.009, 10)
    start_dt = datetime(2020, 1, 1)
    end_dt = datetime(2020, 5, 31)
    daily_weather = Daily(nyc, start=start_dt, end=end_dt).fetch().reset_index()

if hourly_weather_path is not None:
    hourly_weather = pd.read_csv(hourly_weather_path)
    print(f"Loaded hourly weather from local file: {hourly_weather_path}")
else:
    hourly_weather = None
    print("Hourly weather file not found. Proceeding with daily weather only.")

### 3.3 Clean the daily weather table
The raw Meteostat daily file is not directly ready for modeling, so we standardize date keys, coerce numeric weather measures, restrict the analysis period, and impute remaining daily gaps with medians.

In [ ]:
weather = daily_weather.copy()

if "time" in weather.columns and "date" not in weather.columns:
    weather = weather.rename(columns={"time": "date"})
if "datetime" in weather.columns and "date" not in weather.columns:
    weather["datetime"] = pd.to_datetime(weather["datetime"], errors="coerce")
    weather["date"] = weather["datetime"].dt.floor("D")
if "temp" in weather.columns and "tavg" not in weather.columns:
    weather = weather.rename(columns={"temp": "tavg"})

weather["date"] = pd.to_datetime(weather["date"], errors="coerce").dt.floor("D")
weather = weather.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")

daily_candidate_cols = ["tavg", "tmin", "tmax", "prcp", "snow", "wdir", "wspd", "wpgt", "pres", "tsun"]
daily_keep_cols = ["date"] + [c for c in daily_candidate_cols if c in weather.columns]
weather = weather[daily_keep_cols].copy()

for c in [x for x in weather.columns if x != "date"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

weather = weather[(weather["date"] >= "2020-01-01") & (weather["date"] <= "2020-05-31")]

### 3.4 Engineer richer daily climate features from the hourly weather file
To strengthen the weather representation for an A-range submission, convert the hourly NYC weather file into daily aggregate signals that are still compatible with the required date-level merge. These engineered variables capture intraday variability and weather intensity that the plain daily table misses.

In [ ]:
hourly_feature_cols = []

if hourly_weather is not None:
    hw = hourly_weather.copy()

    if "time" in hw.columns and "datetime" not in hw.columns:
        hw = hw.rename(columns={"time": "datetime"})
    hw["datetime"] = pd.to_datetime(hw["datetime"], errors="coerce")
    hw["date"] = hw["datetime"].dt.floor("D")
    hw = hw.dropna(subset=["datetime", "date"]).sort_values("datetime")

    hourly_numeric_cols = ["temp", "dwpt", "rhum", "prcp", "snow", "wdir", "wspd", "wpgt", "pres", "tsun", "coco"]
    for c in [x for x in hourly_numeric_cols if x in hw.columns]:
        hw[c] = pd.to_numeric(hw[c], errors="coerce")

    hourly_daily = hw.groupby("date").agg(
        htemp_mean=("temp", "mean"),
        htemp_std=("temp", "std"),
        hdwpt_mean=("dwpt", "mean"),
        hrhum_mean=("rhum", "mean"),
        hprcp_sum=("prcp", "sum"),
        hprcp_hours=("prcp", lambda s: float((s.fillna(0) > 0).sum())),
        hsnow_sum=("snow", "sum"),
        hwspd_mean=("wspd", "mean"),
        hwpgt_max=("wpgt", "max"),
        hpres_mean=("pres", "mean"),
        hpres_std=("pres", "std"),
        htsun_sum=("tsun", "sum"),
        hcoco_mode=("coco", lambda s: s.mode().iloc[0] if not s.dropna().empty else np.nan),
    ).reset_index()

    # Derived climate interaction signals
    if {"htemp_mean", "hrhum_mean"}.issubset(hourly_daily.columns):
        hourly_daily["heat_humidity_index"] = hourly_daily["htemp_mean"] * hourly_daily["hrhum_mean"]
    if {"htemp_mean", "hdwpt_mean"}.issubset(hourly_daily.columns):
        hourly_daily["temp_dew_gap"] = hourly_daily["htemp_mean"] - hourly_daily["hdwpt_mean"]

    hourly_daily = hourly_daily[(hourly_daily["date"] >= "2020-01-01") & (hourly_daily["date"] <= "2020-05-31")]

    for c in [x for x in hourly_daily.columns if x != "date"]:
        hourly_daily[c] = pd.to_numeric(hourly_daily[c], errors="coerce")

    hourly_feature_cols = [c for c in hourly_daily.columns if c != "date"]
    print("Hourly-derived weather features:", hourly_feature_cols)
    display(hourly_daily.head())
else:
    hourly_daily = pd.DataFrame({"date": weather["date"].drop_duplicates().sort_values()})
    print("No hourly weather file available; hourly-derived features skipped.")

### 3.5 Combine daily weather with engineered hourly weather features
Merge the cleaned daily table with the hourly-derived daily signals, remove empty columns, and impute remaining numeric gaps using weather-only medians before any taxi merge occurs.

In [ ]:
weather = weather.merge(hourly_daily, on="date", how="left")

all_null_cols = [c for c in weather.columns if c != "date" and weather[c].isna().all()]
weather = weather.drop(columns=all_null_cols)

for c in [x for x in weather.columns if x != "date"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")
    weather[c] = weather[c].fillna(weather[c].median())

print("Combined weather shape:", weather.shape)
print("Weather date range:", weather["date"].min(), "to", weather["date"].max())
print("Weather columns used:", [c for c in weather.columns if c != "date"])
display(weather.head())

## 4. Merge taxi + weather and prepare the modeling dataset

Project 2 requires **one combined dataset** made from taxi features plus climate features.  
This notebook keeps the rubric intact while improving local performance:

- build a single taxi + weather modeling dataset
- cache the ready-to-model frame so future reruns can skip reconstruction
- use a time-aware split for TensorFlow regression
- compare the required **Linear**, **MLP**, and **DNN** models with the required loss / optimizer combinations

### 4.1 Define feature lists and month-prep helper
Create reusable logic for weather merge, type coercion, and missing-value handling.

#### 4.1.1 Reload month frames when needed
If cells were rerun and cleanup removed the month DataFrames, reload them from parquet.

In [12]:
import gc

model_cache_path = CACHE_DIR / f"taxi_weather_model_df_{MAX_ROWS if MAX_ROWS is not None else 'all'}rows.parquet"
use_cached_model_df = LOAD_READY_DATASET_IF_AVAILABLE and model_cache_path.exists()

if use_cached_model_df:
    model_df = pd.read_parquet(model_cache_path)
    print("Loaded cached ready-to-model dataset:", model_cache_path.resolve())
else:
    if "jan" not in globals() or "mar" not in globals() or "may" not in globals():
        jan = pd.read_parquet(jan_path)
        mar = pd.read_parquet(mar_path)
        may = pd.read_parquet(may_path)

#### 4.1.2 Define model feature groups
Set taxi features, target variable, and weather features used for downstream training.

In [13]:
taxi_feature_cols = [
    "VendorID", "RatecodeID", "PULocationID", "DOLocationID", "payment_type",
    "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount",
    "pickup_hour", "pickup_weekday", "pickup_day", "pickup_month",
]
target_col = "trip_duration_min"

weather_feature_cols = [c for c in weather.columns if c != "date" and weather[c].notna().any()]
weather_medians = {c: pd.to_numeric(weather[c], errors="coerce").median() for c in weather_feature_cols}

#### 4.1.3 Create month-to-model transformation helper
This helper merges daily weather into taxi rows, enforces numeric dtypes, and drops rows missing required fields.

In [14]:
def build_month_model_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(weather[["date"] + weather_feature_cols], left_on="pickup_date", right_on="date", how="left")

    for col in weather_feature_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(weather_medians[col])

    keep_cols = taxi_feature_cols + weather_feature_cols + [target_col, "pickup_date"]
    out = out.loc[:, keep_cols]

    for col in taxi_feature_cols + weather_feature_cols + [target_col]:
        out[col] = pd.to_numeric(out[col], errors="coerce", downcast="float")

    # Keep rows for explicit data-quality decisions before modeling.
    out = out.loc[out["pickup_date"].notna()].copy()
    return out

### 4.2 Build monthly model-ready frames
Apply the same processing logic independently to Jan/Mar/May.

In [15]:
if not use_cached_model_df:
    jan_model = build_month_model_df(jan)
    mar_model = build_month_model_df(mar)
    may_model = build_month_model_df(may)

    print("Jan model frame:", jan_model.shape)
    print("Mar model frame:", mar_model.shape)
    print("May model frame:", may_model.shape)
else:
    print("Monthly frame build skipped because cached modeling dataset was loaded.")

Jan model frame: (6240894, 24)
Mar model frame: (2922596, 24)
May model frame: (278658, 24)


### 4.3 Concatenate and sort chronologically
Build final modeling frame and create hour-level time proxy for stable time split.

In [16]:
if not use_cached_model_df:
    for _v in ["jan", "mar", "may"]:
        if _v in globals():
            del globals()[_v]
    gc.collect()

    model_df = pd.concat([jan_model, mar_model, may_model], ignore_index=True, copy=False)
    del jan_model, mar_model, may_model
    gc.collect()

    feature_cols = taxi_feature_cols + weather_feature_cols

    # Keep chronological order with minimal overhead
    model_df.sort_values(["pickup_date", "pickup_hour"], kind="mergesort", inplace=True)
    model_df.reset_index(drop=True, inplace=True)

    # Downcast aggressively before sampling/splitting
    for col in feature_cols + [target_col]:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce", downcast="float")

    print("Chronologically ordered modeling dataset shape:", model_df.shape)
else:
    feature_cols = [c for c in model_df.columns if c not in [target_col, "pickup_date"]]
    print("Using cached modeling dataset shape:", model_df.shape)

### 4.4 Data-quality decisions and runtime cap
Inspect missing values and outliers first, then apply explicit preprocessing decisions before the time-aware split.

#### 4.4.1 Configure preprocessing decisions


In [ ]:
APPLY_DATA_QUALITY_DECISIONS = True
FEATURE_IMPUTE_STRATEGY = "median"
OUTLIER_ACTION = "flag_only"  # choose from: flag_only, cap
OUTLIER_IQR_MULTIPLIER = 1.5


#### 4.4.2 Identify modeling features and inspect missing values


In [ ]:
feature_cols = [
    c for c in model_df.columns
    if c not in [target_col, "pickup_date", "pickup_datetime_proxy"]
    and pd.api.types.is_numeric_dtype(model_df[c])
]

continuous_cols = [
    c for c in feature_cols
    if c in [
        "trip_distance", "fare_amount", "tip_amount", "total_amount",
        "passenger_count", "tavg", "tmin", "tmax", "prcp",
        "snow", "wdir", "wspd", "pres",
    ]
]

missing_before = model_df[feature_cols + [target_col]].isna().sum().sort_values(ascending=False)
missing_before_nonzero = missing_before[missing_before > 0]


#### 4.4.3 Apply missing-value and target-validity decisions


In [ ]:
if APPLY_DATA_QUALITY_DECISIONS and FEATURE_IMPUTE_STRATEGY == "median":
    imputed_cells = 0
    for col in feature_cols:
        miss = int(model_df[col].isna().sum())
        if miss > 0:
            fill_val = model_df[col].median()
            if pd.isna(fill_val):
                fill_val = 0.0
            model_df[col] = model_df[col].fillna(fill_val)
            imputed_cells += miss
else:
    imputed_cells = 0

invalid_target_mask = (~np.isfinite(model_df[target_col])) | (model_df[target_col] <= 0)
invalid_target_count = int(invalid_target_mask.sum())
if APPLY_DATA_QUALITY_DECISIONS and invalid_target_count > 0:
    model_df = model_df.loc[~invalid_target_mask].copy()


#### 4.4.4 Audit outliers and finalize the modeling frame


In [ ]:
outlier_rows = []
for col in continuous_cols:
    s = model_df[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        lower, upper = s.min(), s.max()
    else:
        lower = q1 - OUTLIER_IQR_MULTIPLIER * iqr
        upper = q3 + OUTLIER_IQR_MULTIPLIER * iqr

    out_mask = (s < lower) | (s > upper)
    out_count = int(out_mask.sum())
    out_pct = float((out_count / len(s)) * 100) if len(s) else 0.0

    if APPLY_DATA_QUALITY_DECISIONS and OUTLIER_ACTION == "cap" and out_count > 0:
        model_df[col] = s.clip(lower=lower, upper=upper)

    outlier_rows.append({
        "feature": col,
        "iqr_lower": float(lower) if pd.notna(lower) else np.nan,
        "iqr_upper": float(upper) if pd.notna(upper) else np.nan,
        "outlier_count": out_count,
        "outlier_pct": out_pct,
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)

sort_col = "pickup_datetime_proxy" if "pickup_datetime_proxy" in model_df.columns else "pickup_date"
model_df = model_df.sort_values(sort_col).reset_index(drop=True)

if MAX_ROWS is not None and len(model_df) > MAX_ROWS:
    model_df = model_df.iloc[:MAX_ROWS].copy()

missing_after = model_df[feature_cols + [target_col]].isna().sum().sort_values(ascending=False)
missing_after_nonzero = missing_after[missing_after > 0]

viz_df = model_df if len(model_df) <= VIZ_MAX_ROWS else model_df.sample(VIZ_MAX_ROWS, random_state=RANDOM_SEED)


#### 4.4.5 Review the data-quality summary


In [ ]:
readiness_summary = pd.DataFrame([
    {"metric": "rows_after_decisions", "value": len(model_df)},
    {"metric": "invalid_target_rows_removed", "value": invalid_target_count},
    {"metric": "feature_cells_imputed", "value": imputed_cells},
    {"metric": "outlier_action", "value": OUTLIER_ACTION},
])

display(readiness_summary)

print("Missing values before decisions:")
if len(missing_before_nonzero) > 0:
    display(missing_before_nonzero.rename("missing_count").to_frame())
else:
    print("None")

print("Missing values after decisions:")
if len(missing_after_nonzero) > 0:
    display(missing_after_nonzero.rename("missing_count").to_frame())
else:
    print("None")

print("Top outlier rates by feature:")
display(outlier_report.head(15))

print("Combined modeling dataset shape:", model_df.shape)
print("Visualization sample shape:", viz_df.shape)
print("Number of features:", len(feature_cols))
print("Weather cols used:", weather_feature_cols)
print("Pickup date range:", model_df["pickup_date"].min(), "to", model_df["pickup_date"].max())
display(model_df.head(5))


#### 4.4.6 Cache the prepared modeling dataset
Persist the ready-to-model frame so reruns can skip the expensive merge step.


In [ ]:

model_cache_path = CACHE_DIR / f"taxi_weather_model_df_{RUN_MODE}.parquet"

if SAVE_MODELING_ARTIFACTS:
    model_df.to_parquet(model_cache_path, index=False)
    print("Saved ready-to-model dataset to:", model_cache_path.resolve())
else:
    print("Skipping model dataset cache save.")


## 5. Pre-modeling validation, data-quality review, visualizations, and time-aware 80/20 split

### 5.1 Dataset sanity table
Confirm row/column counts and inspect both head and random samples.

In [ ]:
from IPython.display import display

print("Rows:", len(model_df), "Cols:", model_df.shape[1])
display(model_df.head(20).style.format(precision=3))
if len(viz_df) > 0:
    display(viz_df.sample(min(20, len(viz_df)), random_state=RANDOM_SEED).style.format(precision=3))


### 5.2 Temporal trend
Plot median trip duration by pickup date to visualize seasonality and trend.

In [ ]:
daily_duration = model_df.groupby("pickup_date", as_index=False)["trip_duration_min"].median()
plt.figure(figsize=(11, 4))
plt.plot(daily_duration["pickup_date"], daily_duration["trip_duration_min"], linewidth=1.8)
plt.title("Median Trip Duration Over Time (Daily)")
plt.xlabel("Date")
plt.ylabel("Median Trip Duration (min)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### 5.3 Weather-to-target correlation
Visual check of linear relationships between weather features and trip duration.

In [ ]:
plot_weather_cols = [c for c in ["tavg", "tmin", "tmax", "prcp", "snow", "wspd", "pres"] if c in viz_df.columns]
if plot_weather_cols:
    corr_df = viz_df[plot_weather_cols + ["trip_duration_min"]].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Correlation Heatmap: Weather + Trip Duration")
    plt.tight_layout()
    plt.show()


candidate_plot_weather_cols = [
    "tavg", "tmin", "tmax", "prcp", "snow", "wspd", "pres",
    "htemp_mean", "htemp_std", "hrhum_mean", "hprcp_sum",
    "hprcp_hours", "hpres_mean", "heat_humidity_index"
]
plot_weather_cols = [c for c in candidate_plot_weather_cols if c in viz_df.columns]

if plot_weather_cols:
    corr_df = viz_df[plot_weather_cols + ["trip_duration_min"]].corr(numeric_only=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_df, annot=False, cmap="coolwarm", square=False)
    plt.title("Correlation Heatmap: Weather + Trip Duration")
    plt.tight_layout()
    plt.show()

    corr_to_target = (
        corr_df["trip_duration_min"]
        .drop(index="trip_duration_min")
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )
    display(corr_to_target.to_frame("corr_with_trip_duration_min"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
viz_df["PULocationID"].dropna().astype(int).value_counts().head(12).sort_values().plot(kind="barh", ax=axes[0], color="#4e79a7")
axes[0].set_title("Top 12 Pickup Location IDs")
axes[0].set_xlabel("Trip count")

viz_df["DOLocationID"].dropna().astype(int).value_counts().head(12).sort_values().plot(kind="barh", ax=axes[1], color="#f28e2b")
axes[1].set_title("Top 12 Dropoff Location IDs")
axes[1].set_xlabel("Trip count")
plt.tight_layout()
plt.show()


### 5.5 Geographic intensity graph
Approximate zone centroids from NYC taxi geojson and map pickup/dropoff intensity.

#### 5.5.1 Build helper functions for taxi-zone centroids
Download NYC taxi zone geometry and compute lightweight centroid coordinates for visualization.

In [ ]:
def _flatten_coords(geom_type, coords):
    points = []
    if geom_type == "Polygon":
        for ring in coords:
            points.extend(ring)
    elif geom_type == "MultiPolygon":
        for poly in coords:
            for ring in poly:
                points.extend(ring)
    return points


def load_zone_centroids_geojson(url: str = "https://raw.githubusercontent.com/dwillis/nyc-maps/master/taxizones.geojson") -> pd.DataFrame:
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))

    rows = []
    for feat in data.get("features", []):
        props = feat.get("properties", {})
        geom = feat.get("geometry", {})
        loc_id = props.get("location_id") or props.get("LocationID") or props.get("OBJECTID")
        if loc_id is None:
            continue

        pts = _flatten_coords(geom.get("type"), geom.get("coordinates", []))
        if not pts:
            continue

        lons = [p[0] for p in pts if isinstance(p, (list, tuple)) and len(p) >= 2]
        lats = [p[1] for p in pts if isinstance(p, (list, tuple)) and len(p) >= 2]
        if not lons or not lats:
            continue

        rows.append({
            "LocationID": int(loc_id),
            "lon": float(np.mean(lons)),
            "lat": float(np.mean(lats)),
        })

    return pd.DataFrame(rows).drop_duplicates(subset=["LocationID"])

#### 5.5.2 Plot geographic pickup and dropoff intensity
Create side-by-side scatter plots for top pickup and dropoff zones using centroid approximations.

In [ ]:
try:
    zone_centroids = load_zone_centroids_geojson()

    pu_counts = viz_df["PULocationID"].dropna().astype(int).value_counts().rename_axis("LocationID").reset_index(name="pickup_count")
    do_counts = viz_df["DOLocationID"].dropna().astype(int).value_counts().rename_axis("LocationID").reset_index(name="dropoff_count")

    geo_pu = pu_counts.merge(zone_centroids, on="LocationID", how="inner").head(80)
    geo_do = do_counts.merge(zone_centroids, on="LocationID", how="inner").head(80)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sc1 = axes[0].scatter(
        geo_pu["lon"], geo_pu["lat"],
        s=np.clip(geo_pu["pickup_count"] / 400, 20, 300),
        c=geo_pu["pickup_count"], cmap="viridis", alpha=0.8, edgecolor="k", linewidth=0.2,
    )
    axes[0].set_title("NYC Geographic Pickup Intensity (by Taxi Zone)")
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")
    plt.colorbar(sc1, ax=axes[0], fraction=0.046, pad=0.04, label="Trips")

    sc2 = axes[1].scatter(
        geo_do["lon"], geo_do["lat"],
        s=np.clip(geo_do["dropoff_count"] / 400, 20, 300),
        c=geo_do["dropoff_count"], cmap="plasma", alpha=0.8, edgecolor="k", linewidth=0.2,
    )
    axes[1].set_title("NYC Geographic Dropoff Intensity (by Taxi Zone)")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    plt.colorbar(sc2, ax=axes[1], fraction=0.046, pad=0.04, label="Trips")

    plt.tight_layout()
    plt.show()

except Exception as geo_err:
    print(f"Geographic visualization skipped (lookup unavailable): {geo_err}")


### 5.6 Time-aware 80/20 split and scaling
Split chronologically to avoid leakage, then standardize features.

In [ ]:
split_idx = int(len(model_df) * 0.8)

X_all = model_df[feature_cols].to_numpy(dtype=np.float32, copy=True)
y_all = model_df[target_col].to_numpy(dtype=np.float32, copy=True)

train_dates = model_df.iloc[:split_idx]["pickup_date"]
val_dates = model_df.iloc[split_idx:]["pickup_date"]

# Release visualization frame before scaling/training
del viz_df
gc.collect()

X_train = X_all[:split_idx]
X_val = X_all[split_idx:]
y_train = y_all[:split_idx]
y_val = y_all[split_idx:]

del X_all, y_all
gc.collect()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32, copy=False)
X_val_scaled = scaler.transform(X_val).astype(np.float32, copy=False)

if SAVE_MODELING_ARTIFACTS:
    np.save(CACHE_DIR / f"X_train_scaled_{RUN_MODE}.npy", X_train_scaled)
    np.save(CACHE_DIR / f"X_val_scaled_{RUN_MODE}.npy", X_val_scaled)
    np.save(CACHE_DIR / f"y_train_{RUN_MODE}.npy", y_train)
    np.save(CACHE_DIR / f"y_val_{RUN_MODE}.npy", y_val)
    print("Saved scaled arrays to cache directory.")

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_scaled, y_train))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val_scaled, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Modeling dataset shape:", model_df.shape)
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Training date range:", train_dates.min(), "to", train_dates.max())
print("Validation date range:", val_dates.min(), "to", val_dates.max())
print("Approx X_train_scaled memory (GB):", round(X_train_scaled.nbytes / 1024**3, 3))
print("Approx X_val_scaled memory (GB):", round(X_val_scaled.nbytes / 1024**3, 3))

## 6. TensorFlow model definitions

We define the three required TensorFlow/Keras models:

1. **Linear Regression** – no hidden layers  
2. **MLP** – moderate hidden architecture  
3. **DNN** – deeper network with at least 2 hidden layers


### 6.1 Define TensorFlow model builders
Create Linear, MLP, and DNN architectures required by the project prompt.

#### 6.1.1 Linear regression baseline (Keras Sequential, no hidden layers)
This is the assignment-required linear baseline for comparison against deeper models.

In [ ]:
def build_linear_model(input_dim: int) -> keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(1)
    ], name="Linear_Regression")
    return model

#### 6.1.2 MLP model (compact non-linear baseline)
Two hidden layers with dropout for regularized regression learning.

In [ ]:
def build_mlp_model(input_dim: int) -> keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1)
    ], name="MLP")
    return model

#### 6.1.3 DNN model (deeper architecture) and builder registry
Four hidden layers with dropout; model registry lets us train all architectures in a loop.

In [ ]:
def build_dnn_model(input_dim: int) -> keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1)
    ], name="DNN")
    return model

MODEL_BUILDERS = {
    "Linear": build_linear_model,
    "MLP": build_mlp_model,
    "DNN": build_dnn_model,
}

### 6.2 Print model summaries
Verify each architecture before training.

In [ ]:

for name, builder in MODEL_BUILDERS.items():
    model = builder(X_train_scaled.shape[1])
    print(f"\n{name} model summary")
    model.summary()
    del model
    gc.collect()


## 7. Training configuration

### 7.1 Configure Project 2 TensorFlow experiments

#### 7.1.1 Set epoch and monitoring choices


In [ ]:
EPOCHS = DEFAULT_EPOCHS
MONITOR_METRIC = "mae" 


#### 7.1.2 Define the rubric-compliant TensorFlow experiment plan


In [ ]:
EXPERIMENT_PLAN = [
    {"model": "Linear", "optimizer": "SGD",     "learning_rate": 0.0010, "loss_name": "MSE", "loss": "mse"},
    {"model": "Linear", "optimizer": "Adam",    "learning_rate": 0.0005, "loss_name": "MAE", "loss": "mae"},
    {"model": "MLP",    "optimizer": "Adam",    "learning_rate": 0.0005, "loss_name": "MSE", "loss": "mse"},
    {"model": "MLP",    "optimizer": "RMSProp", "learning_rate": 0.0003, "loss_name": "MAE", "loss": "mae"},
    {"model": "DNN",    "optimizer": "SGD",     "learning_rate": 0.0005, "loss_name": "MAE", "loss": "mae"},
    {"model": "DNN",    "optimizer": "RMSProp", "learning_rate": 0.0003, "loss_name": "MSE", "loss": "mse"},
]

required_models = {"Linear", "MLP", "DNN"}
required_optimizers = {"SGD", "Adam", "RMSProp"}
required_losses = {"MSE", "MAE"}
required_learning_rates = sorted({cfg["learning_rate"] for cfg in EXPERIMENT_PLAN})

assert {cfg["model"] for cfg in EXPERIMENT_PLAN} == required_models
assert {cfg["optimizer"] for cfg in EXPERIMENT_PLAN} == required_optimizers
assert {cfg["loss_name"] for cfg in EXPERIMENT_PLAN} == required_losses
assert len(required_learning_rates) >= 2


#### 7.1.3 Build the optimizer factory


In [ ]:
def make_optimizer(name: str, learning_rate: float):
    if name == "SGD":
        return SGD(learning_rate=learning_rate, clipnorm=1.0)
    if name == "Adam":
        return Adam(learning_rate=learning_rate, clipnorm=1.0)
    if name == "RMSProp":
        return RMSprop(learning_rate=learning_rate, clipnorm=1.0)
    raise ValueError(f"Unknown optimizer: {name}")


#### 7.1.4 Review the final TensorFlow run plan


In [ ]:
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Experiment count:", len(EXPERIMENT_PLAN))
display(pd.DataFrame(EXPERIMENT_PLAN))


This run plan is intentionally compact so the notebook can finish on a local laptop while still satisfying the Project 2 rubric. It covers all three required TensorFlow architectures, all three required optimizers, both required regression losses, and multiple learning rates. The learning rates are conservative and optimizers use gradient clipping to reduce unstable runs during 100-epoch training.

In [ ]:
def plot_training_history(history_dict, title: str):
    plt.figure(figsize=(9, 5))
    plt.plot(history_dict["loss"], label="Training Loss")
    plt.plot(history_dict["val_loss"], label="Validation Loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate_predictions(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"mse": mse, "mae": mae, "rmse": rmse, "r2": r2}


## 8. Train all TensorFlow experiments

This section runs a compact but rubric-compliant TensorFlow comparison. Each selected run still trains for 100 epochs in `stable` and `full` mode, and the experiment list collectively covers all required architectures, optimizers, losses, and multiple learning rates.

### 8.1 Train TensorFlow experiments

Run a **reduced but rubric-compliant** experiment plan that preserves:

- Linear / MLP / DNN
- SGD / Adam / RMSProp
- MSE / MAE
- multiple learning rates
- `epoch = 100` in stable/full mode

This keeps the notebook submission-ready while making local execution realistic.

#### 8.1.1 Initialize memory-safe TensorFlow experiment trackers

In [ ]:

tf_results = []
tf_histories = {}
best_tf_run = None
best_tf_row = None
best_tf_metrics = None
best_model_path = CACHE_DIR / "best_tensorflow_model.keras"
if best_model_path.exists():
    best_model_path.unlink()

log_root = CACHE_DIR / "tensorboard_logs"
log_root.mkdir(exist_ok=True)


#### 8.1.2 Run the TensorFlow training grid with safer memory handling

#### 8.1.2A Define a helper to run one TensorFlow experiment


In [ ]:
def run_tensorflow_experiment(cfg: dict):
    model_name = cfg["model"]
    optimizer_name = cfg["optimizer"]
    learning_rate = cfg["learning_rate"]
    loss_name = cfg["loss_name"]
    loss_value = cfg["loss"]
    builder = MODEL_BUILDERS[model_name]

    keras.backend.clear_session()
    gc.collect()

    run_name = f"{model_name}_{optimizer_name}_lr{learning_rate}_{loss_name}"
    print(f"\nTraining: {run_name} | epochs={EPOCHS} | batch_size={BATCH_SIZE}")

    model = builder(X_train_scaled.shape[1])
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss=loss_value,
        metrics=["mae"],
    )

    callbacks = []
    if ENABLE_TENSORBOARD:
        log_dir = log_root / run_name / time.strftime("%Y%m%d-%H%M%S")
        callbacks.append(TensorBoard(log_dir=str(log_dir), histogram_freq=0, write_graph=False))

    start = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        verbose=FIT_VERBOSE,
        callbacks=callbacks,
    )
    elapsed = time.time() - start

    y_pred = model.predict(val_ds, verbose=0).reshape(-1)
    metrics = evaluate_predictions(y_val, y_pred)

    result = {
        "framework": "TensorFlow",
        "run_name": run_name,
        "model": model_name,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "loss_name": loss_name,
        "final_train_loss": float(history.history["loss"][-1]),
        "final_val_loss": float(history.history["val_loss"][-1]),
        "mse": float(metrics["mse"]),
        "mae": float(metrics["mae"]),
        "rmse": float(metrics["rmse"]),
        "r2": float(metrics["r2"]),
        "epochs": int(EPOCHS),
        "batch_size": int(BATCH_SIZE),
        "train_time_sec": float(elapsed),
    }

    history_payload = {
        "loss": [float(x) for x in history.history.get("loss", [])],
        "val_loss": [float(x) for x in history.history.get("val_loss", [])],
        "mae": [float(x) for x in history.history.get("mae", [])],
        "val_mae": [float(x) for x in history.history.get("val_mae", [])],
    }

    return run_name, model, result, metrics, history_payload


#### 8.1.2B Execute the TensorFlow experiment plan


In [ ]:
print("CHECKPOINT 8.1.2: TensorFlow training grid start")

for cfg in EXPERIMENT_PLAN:
    run_name, model, result, metrics, history_payload = run_tensorflow_experiment(cfg)

    tf_results.append(result)
    tf_histories[run_name] = history_payload

    if (best_tf_row is None) or (
        (result["mae"], result["mse"], result["final_val_loss"]) <
        (best_tf_row["mae"], best_tf_row["mse"], best_tf_row["final_val_loss"])
    ):
        best_tf_row = result
        best_tf_run = run_name
        best_tf_metrics = metrics
        model.save(best_model_path)
        print(f"Updated best model -> {best_tf_run}")

    print(
        f"Completed {run_name} | "
        f"val_MAE={result['mae']:.4f} | "
        f"val_MSE={result['mse']:.4f} | "
        f"time={result['train_time_sec']:.1f}s"
    )

    del model, metrics, history_payload
    gc.collect()

print("CHECKPOINT 8.1.2 complete. TensorFlow experiments finished.")


#### 8.1.3 Build and preview TensorFlow results table
Rank candidate runs by MAE and MSE to quickly identify top-performing settings.

In [ ]:
tf_results_df = pd.DataFrame(tf_results).sort_values(
    ["mae", "mse", "final_val_loss", "train_time_sec"]
).reset_index(drop=True)

display(tf_results_df)


## 9. Plot training vs validation loss

Below, we visualize the training and validation loss curves for every TensorFlow run in the final experiment plan.


In [ ]:
for run_name in tf_results_df["run_name"].tolist():
    if run_name in tf_histories:
        plot_training_history(tf_histories[run_name], f"{run_name}: Training vs Validation Loss")


## 10. Select the best TensorFlow model and generate predictions

We select the best TensorFlow model using **lowest validation MAE**, then generate predictions and review performance.


### 10.1 Load the best saved TensorFlow model and compute validation predictions


In [ ]:
if best_model_path.exists():
    best_tf_model = keras.models.load_model(best_model_path)
else:
    raise FileNotFoundError("Best TensorFlow model file was not saved correctly.")

best_tf_preds = best_tf_model.predict(X_val_scaled, verbose=0).reshape(-1)
best_tf_metrics = evaluate_predictions(y_val, best_tf_preds)

print("Best TensorFlow run:", best_tf_run)
print(pd.Series(best_tf_row))


### 10.2 Preview actual and predicted trip durations


In [ ]:
pred_df = pd.DataFrame({
    "actual_trip_duration_min": y_val[:20],
    "predicted_trip_duration_min": best_tf_preds[:20],
})
display(pred_df)


### 10.3 Plot actual vs predicted trip duration


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_val[:1000], best_tf_preds[:1000], alpha=0.25, s=14)
diag_min = float(min(y_val[:1000].min(), best_tf_preds[:1000].min()))
diag_max = float(max(y_val[:1000].max(), best_tf_preds[:1000].max()))
plt.plot([diag_min, diag_max], [diag_min, diag_max], linestyle="--", color="red")
plt.title("Actual vs Predicted Trip Duration")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.tight_layout()
plt.show()


### 10.4 Plot residuals and print validation metrics


In [ ]:
residuals = best_tf_preds - y_val

plt.figure(figsize=(8, 4))
plt.scatter(best_tf_preds[:1000], residuals[:1000], alpha=0.25, s=14)
plt.axhline(0.0, linestyle="--", color="red")
plt.title("Residual Plot for Best TensorFlow Model")
plt.xlabel("Predicted trip duration")
plt.ylabel("Residual")
plt.tight_layout()
plt.show()

print("\nBest TensorFlow validation metrics")
for k, v in best_tf_metrics.items():
    print(f"{k.upper()}: {v:.4f}")


## 11. TensorFlow model comparison summary

In [ ]:
best_per_arch = (
    tf_results_df
    .sort_values(["model", "mae", "mse", "final_val_loss"])
    .groupby("model", as_index=False)
    .first()
    .sort_values("mae")
    .reset_index(drop=True)
)

display(best_per_arch)


## 12. PyTorch extra credit (optional)

### 12.1 PyTorch section is optional and disabled by default

In [ ]:
if ENABLE_PYTORCH_EXTRA_CREDIT:
    class TorchDNN(nn.Module):
        def __init__(self, input_dim: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.ReLU(),
                nn.Dropout(0.30),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.30),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(32, 1),
            )

        def forward(self, x):
            return self.net(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch device:", device)

    torch_train_rows = min(len(X_train_scaled), TORCH_MAX_ROWS)
    torch_val_rows = min(len(X_val_scaled), max(1, int(TORCH_MAX_ROWS * 0.2)))

    X_train_t = torch.tensor(X_train_scaled[:torch_train_rows], dtype=torch.float32)
    y_train_t = torch.tensor(y_train[:torch_train_rows].reshape(-1, 1), dtype=torch.float32)
    X_val_t = torch.tensor(X_val_scaled[:torch_val_rows], dtype=torch.float32)
    y_val_t = torch.tensor(y_val[:torch_val_rows].reshape(-1, 1), dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
else:
    print("PyTorch extra credit is disabled. Set ENABLE_PYTORCH_EXTRA_CREDIT = True to run it.")


## 13. Train the PyTorch DNN (optional extra credit)

In [ ]:
if ENABLE_PYTORCH_EXTRA_CREDIT:
    torch_model = TorchDNN(X_train_scaled.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=0.001)

    torch_history = {"train_loss": [], "val_loss": []}
    start = time.time()
else:
    torch_model = None
    torch_history = None
    torch_elapsed = None


### 13.1 Run PyTorch training loop only when extra credit is enabled

In [ ]:
if ENABLE_PYTORCH_EXTRA_CREDIT:
    for epoch in range(EPOCHS):
        torch_model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = torch_model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        torch_model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = torch_model(xb)
                loss = criterion(preds, yb)
                val_losses.append(loss.item())

        torch_history["train_loss"].append(float(np.mean(train_losses)))
        torch_history["val_loss"].append(float(np.mean(val_losses)))

    torch_elapsed = time.time() - start
else:
    print("Skipping PyTorch training.")


### 13.2 Evaluate PyTorch validation performance if enabled

In [ ]:
if ENABLE_PYTORCH_EXTRA_CREDIT:
    torch_model.eval()
    with torch.no_grad():
        torch_preds = torch_model(X_val_t.to(device)).cpu().numpy().reshape(-1)

    torch_metrics = evaluate_predictions(y_val[:len(torch_preds)], torch_preds)

    print("PyTorch training complete.")
    for k, v in torch_metrics.items():
        print(f"{k.upper()}: {v:.4f}")
    print(f"Training time (sec): {torch_elapsed:.2f}")
else:
    torch_preds = None
    torch_metrics = None


## 14. Plot PyTorch training vs validation loss (optional)

In [ ]:
if ENABLE_PYTORCH_EXTRA_CREDIT:
    plt.figure(figsize=(9, 5))
    plt.plot(torch_history["train_loss"], label="Training Loss")
    plt.plot(torch_history["val_loss"], label="Validation Loss")
    plt.title("PyTorch DNN: Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No PyTorch loss plot because extra credit is disabled.")


## 15. Compare TensorFlow and PyTorch (only if extra credit is enabled)

In [ ]:
comparison_rows = [
    {
        "framework": "TensorFlow",
        "model": best_tf_row["model"],
        "optimizer": best_tf_row["optimizer"],
        "learning_rate": best_tf_row["learning_rate"],
        "loss": best_tf_row["loss_name"],
        "mse": best_tf_metrics["mse"],
        "mae": best_tf_metrics["mae"],
        "rmse": best_tf_metrics["rmse"],
        "r2": best_tf_metrics["r2"],
        "train_time_sec": float(best_tf_row["train_time_sec"]),
    }
]

if ENABLE_PYTORCH_EXTRA_CREDIT and torch_metrics is not None:
    comparison_rows.append(
        {
            "framework": "PyTorch",
            "model": "DNN",
            "optimizer": "Adam",
            "learning_rate": 0.001,
            "loss": "MSE",
            "mse": torch_metrics["mse"],
            "mae": torch_metrics["mae"],
            "rmse": torch_metrics["rmse"],
            "r2": torch_metrics["r2"],
            "train_time_sec": torch_elapsed,
        }
    )

framework_comparison_df = pd.DataFrame(comparison_rows).sort_values("mae").reset_index(drop=True)
display(framework_comparison_df)


## 16. Final conclusions

In [ ]:
print("Best TensorFlow run:", best_tf_run)
print("Best architecture:", best_tf_row["model"])
print("Best optimizer:", best_tf_row["optimizer"])
print("Best learning rate:", best_tf_row["learning_rate"])
print("Best loss function:", best_tf_row["loss_name"])

print("\nTensorFlow best metrics")
for k, v in best_tf_metrics.items():
    print(f"{k.upper()}: {v:.4f}")

if ENABLE_PYTORCH_EXTRA_CREDIT and torch_metrics is not None:
    print("\nPyTorch metrics")
    for k, v in torch_metrics.items():
        print(f"{k.upper()}: {v:.4f}")

print("\nSuggested interpretation guide:")
print("1. Lowest validation MAE is the primary selection criterion for the final TensorFlow model.")
print("2. Compare Linear vs MLP vs DNN to explain whether added nonlinearity improved accuracy.")
print("3. Use the loss curves to discuss convergence and possible overfitting.")
print("4. Use the optional TensorFlow vs PyTorch table only if you enable extra credit.")
